# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

## Loading Data

We begin by loading the interaction data.

In [2]:
df = pd.read_csv("interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


In [3]:
df

,user_id,item_id
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
55370,423,53
55371,485,855
55372,801,333
55373,662,853


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [5]:
import random
#def sample_negatives(df, num_neg=1):
    # TODO:
    # 1. Create a set of (user, item) interactions
    # 2. For each positive interaction:
    #    - add (user, item, 1)
    #    - sample 'num_neg' negative items
    # 3. Return a dataframe with columns: user, item, label

def sample_negatives(df,num_neg=1):
    useritemset=set(zip(df["user_id"],df["item_id"]))
    all_items=df["item_id"].unique().tolist()
    sampled=[]

    for user,item in zip(df["user_id"],df["item_id"]):
        # positive sample
        sampled.append((user,item,1))
        # negative samples
        for _ in range(num_neg):
            neg_item = random.choice(all_items)
            while (user,neg_item) in useritemset:
                neg_item=random.choice(all_items)
            sampled.append((user,neg_item, 0))
    sampled_df = pd.DataFrame(sampled,columns=["user","item","label"])
    return sampled_df

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [6]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user'].values)
        self.items = torch.tensor(df['item'].values)
        self.labels = torch.tensor(df['label'].values).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [7]:
train_data = sample_negatives(train_df)

train_loader = DataLoader(InteractionDataset(train_data), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

Ans : Negative sampling is used in training because recommendation datasets usually contain only positive interactions (items users interacted with) and very few explicit negative interactions. By sampling unseen items as negatives, the model learns to distinguish between preferred and non-preferred items. During evaluation, negatives are only used to test ranking performance, not for learning or updating model parameters.

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [8]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [9]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        self.user_embedding=nn.Embedding(num_users,emb_dim)
        # - Define item embedding layer
        self.item_embedding=nn.Embedding(num_items,emb_dim)

    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        a=self.user_embedding(user)
        # - Compute dot product
        b=self.item_embedding(item)
        # - Return a single score per pair
        score=(a*b).sum(dim=1)
        return score

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [10]:
def train(model, dataloader, epochs):
    # TODO:
    # - Move model to device (optional)
    # - Define optimizer
    # - Define loss function
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
    criterion=nn.BCEWithLogitsLoss()

    losses = []  # track loss per epoch
    printed=False
    for epoch in range(epochs):
        total_loss = 0

        for user, item, label in dataloader:
            # TODO:
            # - Forward pass
            # - Compute loss
            # - Backpropagation
            # - Optimizer step

            user=user.to(device)
            item=item.to(device)
            label=label.float().to(device)
            
            optimizer.zero_grad()
            pred=model(user,item)
            if not printed:
                print(pred.mean().item(), pred.std().item())
                printed = True
            loss=criterion(pred,label)
            loss.backward()
            optimizer.step()
            total_loss+=loss.item()

        losses.append(total_loss)
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    return losses

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [11]:
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total

# TODO:
num_users=df['user_id'].nunique()
num_items=df['item_id'].nunique()
# 1. Initialize MF model
model=MF(num_users,num_items,emb_dim=64)
# 2. Train it
train(model,train_loader,epochs=20)
# 3. Evaluate it
hr=hit_at_k(model,test_df,df,K=10,num_neg=100)
print(hr)

0.7946467399597168 7.6143798828125
Epoch 1, Loss: 1117.8462
Epoch 2, Loss: 1035.0187
Epoch 3, Loss: 955.3710
Epoch 4, Loss: 882.8027
Epoch 5, Loss: 815.0319
Epoch 6, Loss: 752.2688
Epoch 7, Loss: 692.5195
Epoch 8, Loss: 638.0923
Epoch 9, Loss: 587.5060
Epoch 10, Loss: 540.8854
Epoch 11, Loss: 496.4784
Epoch 12, Loss: 455.9414
Epoch 13, Loss: 417.5605
Epoch 14, Loss: 383.2539
Epoch 15, Loss: 349.9576
Epoch 16, Loss: 319.4644
Epoch 17, Loss: 292.1521
Epoch 18, Loss: 265.9495
Epoch 19, Loss: 241.4619
Epoch 20, Loss: 219.6494
0.11429345372460496


If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [12]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        self.user_embedding=nn.Embedding(num_users,emb_dim)
        # - Define item embedding layer
        self.item_embedding=nn.Embedding(num_items,emb_dim)
        # - Define MLP layers
        self.mlp=nn.Sequential(
            nn.Linear(2*emb_dim , 256),
            nn.ReLU(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        # - Concatenate user and item embeddings
        # - Pass through MLP
        # - Return a single score
        u=self.user_embedding(user)
        i=self.item_embedding(item)
        x=torch.cat([u,i],dim=1)
        score=self.mlp(x)
        return score.squeeze()

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [13]:
# TODO:
num_users=df['user_id'].nunique()
num_items=df['item_id'].nunique()
# 1. Initialize MLP model
model=MLP(num_users,num_items,emb_dim=64)
# 2. Train it
train(model,train_loader,epochs=20)
# 3. Evaluate it
hr=hit_at_k(model,test_df,df,K=10,num_neg=100)
print(hr)

0.050805773586034775 0.02480928599834442
Epoch 1, Loss: 209.3060
Epoch 2, Loss: 173.9583
Epoch 3, Loss: 164.1245
Epoch 4, Loss: 157.2478
Epoch 5, Loss: 150.1075
Epoch 6, Loss: 142.2360
Epoch 7, Loss: 133.1131
Epoch 8, Loss: 122.7629
Epoch 9, Loss: 113.1267
Epoch 10, Loss: 101.9345
Epoch 11, Loss: 90.4670
Epoch 12, Loss: 79.0746
Epoch 13, Loss: 69.5299
Epoch 14, Loss: 61.0148
Epoch 15, Loss: 54.9170
Epoch 16, Loss: 48.9126
Epoch 17, Loss: 41.5685
Epoch 18, Loss: 37.1808
Epoch 19, Loss: 32.2088
Epoch 20, Loss: 31.2277
0.5090744920993228


## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

Ans : The Matrix Factorization (MF) model achieved a Hit@10 score of approximately 0.25, while the MLP-based recommender achieved approximately 0.57.

Ans : The MLP-based recommender performed significantly better than the MF model.

---

### Think & Reflect

- Why might the MLP model outperform MF?

Ans : MF only models linear interactions between user and item embeddings using a dot product. The MLP model can learn more complex nonlinear relationships between users and items through multiple neural network layers and activation functions.

- In what cases might MF perform just as well or better?

Ans : MF can perform well on smaller datasets, sparse datasets, or simpler recommendation tasks. It is also computationally cheaper and easier to train, so it may generalize better when data is limited.

- How does embedding size affect performance?

Ans : Larger embedding dimensions allow the model to capture more detailed user-item relationships, which can improve performance. However, very large embeddings may increase overfitting and computational cost.

- Did you observe any signs of overfitting?

Ans : The MLP loss showed some fluctuations at later epochs, which may indicate mild overfitting or instability as the model became highly fitted to the training data.

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [14]:
class NCF(nn.Module):

    def __init__(self, num_users, num_items, emb_dim):

        super().__init__()

        # MF embeddings
        self.mf_user = nn.Embedding(num_users, emb_dim)
        self.mf_item = nn.Embedding(num_items, emb_dim)

        # MLP embeddings
        self.mlp_user = nn.Embedding(num_users, emb_dim)
        self.mlp_item = nn.Embedding(num_items, emb_dim)

        # Better MLP branch
        self.mlp = nn.Sequential(

            nn.Linear(2 * emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU()
        )

        # Final layer
        self.final = nn.Sequential(

            nn.Linear(emb_dim + 64, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, user, item):

        # MF branch
        u_mf = self.mf_user(user)
        i_mf = self.mf_item(item)

        mf_vec = u_mf * i_mf

        # MLP branch
        u_mlp = self.mlp_user(user)
        i_mlp = self.mlp_item(item)

        x = torch.cat([u_mlp, i_mlp], dim=1)

        mlp_vec = self.mlp(x)

        # Combine
        combined = torch.cat([mf_vec, mlp_vec], dim=1)

        score = self.final(combined)

        return score.squeeze()

In [15]:
model = NCF(num_users, num_items, emb_dim=64)

train(model, train_loader, epochs=20)

hr = hit_at_k(model, test_df, df)

print(hr)

0.04331837221980095 0.11711092293262482
Epoch 1, Loss: 215.6350
Epoch 2, Loss: 180.1885
Epoch 3, Loss: 170.1517
Epoch 4, Loss: 163.9549
Epoch 5, Loss: 158.1443
Epoch 6, Loss: 151.7947
Epoch 7, Loss: 144.4663
Epoch 8, Loss: 136.2096
Epoch 9, Loss: 127.2534
Epoch 10, Loss: 118.2381
Epoch 11, Loss: 108.8192
Epoch 12, Loss: 99.8434
Epoch 13, Loss: 89.8847
Epoch 14, Loss: 81.3134
Epoch 15, Loss: 73.5533
Epoch 16, Loss: 65.8036
Epoch 17, Loss: 58.1612
Epoch 18, Loss: 52.2076
Epoch 19, Loss: 45.9705
Epoch 20, Loss: 39.7414
0.5136794582392776


In this experiment, combining MF and MLP improved performance compared to the MF model and slightly improved over the standalone MLP model. The hybrid NCF model benefited from both linear interaction modeling through MF and nonlinear interaction modeling through the MLP branch.

Using separate embeddings for the MF and MLP components allowed each branch to learn different types of user-item relationships more effectively. The model balanced both components by concatenating the MF and MLP representations and learning a final prediction layer instead of directly adding their scores.

This experiment suggests that combining both approaches can improve recommendation quality when properly balanced.